<a href="https://colab.research.google.com/github/shyampandey263/langchain-rag-qa/blob/main/langchain-rag-qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama

Selecting previously unselected package pci.ids.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package lshw.
Preparing to unpack .../2-lshw_02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../3-pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselected package usb.ids.
Preparing to unpack .../4-usb.ids_2024.03.18-1_all.deb ...
Unpacking usb.ids (2024.03.18-1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../5-

In [25]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull llama3.1
!ollama pull nomic-embed-text

In [3]:
!ollama list


NAME                       ID              SIZE      MODIFIED      
nomic-embed-text:latest    0a109f422b47    274 MB    2 minutes ago    
llama3.1:latest            46e0c10c039e    4.9 GB    2 minutes ago    


In [4]:
!pip install -q langchain langchain-ollama langchain-community chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

In [5]:
import langchain, chromadb
print("langchain:", langchain.__version__)
print("chromadb:", chromadb.__version__)

langchain: 1.3.18
chromadb: 1.5.9


In [1]:
%%writefile doc1.txt
Petrol prices in India are revised daily based on international crude oil prices,
foreign exchange rates, and the trade parity price mechanism. Oil Marketing
Companies (OMCs) such as IOCL, BPCL, and HPCL update retail prices at 6 AM daily.

Writing doc1.txt


In [4]:
%%writefile doc2.txt
The Petroleum Planning and Analysis Cell (PPAC) is an attached office under the
Ministry of Petroleum and Natural Gas. It compiles data on petroleum product
consumption, pricing, and import/export trends across Indian states.

Overwriting doc2.txt


In [5]:
%%writefile doc3.txt
Diesel consumption in India is closely tied to the transport and agriculture
sectors. Seasonal demand spikes occur during harvest season and around major
festivals due to increased logistics activity.

Writing doc3.txt


In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

docs = []
for path in ["doc1.txt", "doc2.txt", "doc3.txt"]:
    docs.extend(TextLoader(path).load())

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Loaded {len(docs)} documents, split into {len(chunks)} chunks.")

ModuleNotFoundError: No module named 'langchain_community'

In [7]:
!pip show langchain langchain-community langchain-ollama chromadb 2>&1 | grep -E "Name|WARNING|not found"

Name: langchain


In [8]:
!pip install -q langchain langchain-community langchain-ollama chromadb langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

In [9]:
import langchain_community, langchain_ollama, chromadb
print("langchain_community OK")
print("langchain_ollama OK")
print("chromadb OK")

/tmp/ipykernel_2858/1560880410.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community, langchain_ollama, chromadb


langchain_community OK
langchain_ollama OK
chromadb OK


In [22]:
import subprocess, time
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# Ensure Ollama server is running
subprocess.Popen(["ollama", "serve"])
time.sleep(10) # Increased sleep to ensure server is fully up

docs = []
for path in ["doc1.txt", "doc2.txt", "doc3.txt"]:
    docs.extend(TextLoader(path).load())

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Loaded {len(docs)} documents, split into {len(chunks)} chunks.")

Loaded 3 documents, split into 3 chunks.


In [21]:
!ollama pull nomic-embed-text

In [14]:
!which ollama

In [15]:
!ls -l /usr/local/bin/

total 268728
-rwxr-xr-x 1 root   root         316 Sep  4 13:40 accelerate
-rwxr-xr-x 1 root   root         308 Sep  4 13:40 accelerate-config
-rwxr-xr-x 1 root   root         310 Sep  4 13:40 accelerate-estimate-memory
-rwxr-xr-x 1 root   root         308 Sep  4 13:40 accelerate-launch
-rwxr-xr-x 1 root   root         307 Sep  4 13:40 accelerate-merge-weights
-rwxr-xr-x 1 root   root         296 Sep  4 13:40 adk
-rwxr-xr-x 1 root   root   224268544 Sep 21 13:22 agy
-rwxr-xr-x 1 root   root         295 Sep  4 13:40 apsw
-rwxr-xr-x 1 root   root         299 Sep  4 13:40 b2view
-rwxr-xr-x 1 root   root        1089 Sep  4 13:33 beeline
-rwxr-xr-x 1 root   root        1064 Sep  4 13:33 beeline.cmd
-rwxr-xr-x 1 root   root         296 Sep  4 13:40 bokeh
-rwxr-xr-x 1 root   root         291 Sep  4 13:40 ccmake
-rwxr-xr-x 1 root   root         298 Sep  4 13:40 cffi-gen-src
-rwxr-xr-x 1 root   root         304 Sep  4 13:40 chardetect
-rwxr-xr-x 1 root   root         213 Sep 23 10:34 chroma
-rwx

In [16]:
!ls -l /usr/local/

total 56
drwxr-xr-x 1 ubuntu ubuntu 4096 Sep 23 10:34 bin
drwxr-xr-x 3 root   root   4096 Sep 21 13:17 colab
lrwxrwxrwx 1 root   root     22 Mar 10  2025 cuda -> /etc/alternatives/cuda
lrwxrwxrwx 1 root   root     25 Mar 10  2025 cuda-12 -> /etc/alternatives/cuda-12
drwxr-xr-x 1 root   root   4096 Mar 10  2025 cuda-12.8
drwxr-xr-x 1 ubuntu ubuntu 4096 Sep  4 13:40 etc
drwxr-xr-x 2 root   root   4096 Jan 27  2025 games
drwxr-xr-x 1 ubuntu ubuntu 4096 Sep  4 13:40 include
drwxr-xr-x 1 ubuntu ubuntu 4096 Sep  4 13:40 lib
drwxr-xr-x 3 ubuntu ubuntu 4096 Apr  9 20:16 libexec
-rw-r--r-- 1 ubuntu ubuntu 1321 Apr  9 20:16 LICENSE.md
lrwxrwxrwx 1 root   root      9 Jan 27  2025 man -> share/man
drwxr-xr-x 3 root   root   4096 Sep  4 13:40 opt
drwxr-xr-x 2 root   root   4096 Jan 27  2025 sbin
drwxr-xr-x 1 ubuntu ubuntu 4096 Sep  4 13:40 share
drwxr-xr-x 2 root   root   4096 Jan 27  2025 src


In [17]:
!ls -l

total 16
-rw-r--r-- 1 root root  240 Sep 23 10:29 doc1.txt
-rw-r--r-- 1 root root  226 Sep 23 10:30 doc2.txt
-rw-r--r-- 1 root root  201 Sep 23 10:30 doc3.txt
drwxr-xr-x 1 root root 4096 Sep  4 13:32 sample_data


In [18]:
!find / -name ollama 2>/dev/null

/usr/local/lib/python3.13/dist-packages/ollama


In [11]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [23]:
results = retriever.invoke("What does PPAC do?")
for r in results:
    print(r.page_content)
    print("---")

The Petroleum Planning and Analysis Cell (PPAC) is an attached office under the
Ministry of Petroleum and Natural Gas. It compiles data on petroleum product
consumption, pricing, and import/export trends across Indian states.
---
Petrol prices in India are revised daily based on international crude oil prices,
foreign exchange rates, and the trade parity price mechanism. Oil Marketing
Companies (OMCs) such as IOCL, BPCL, and HPCL update retail prices at 6 AM daily.
---


In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1", temperature=0)

prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If the context doesn't contain the answer, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(retrieved_docs):
    return "\n\n".join(d.page_content for d in retrieved_docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What does PPAC do?"))

PPAC compiles data on petroleum product consumption, pricing, and import/export trends across Indian states.


In [28]:
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_petroleum_docs(question: str) -> str:
    """Answer questions about Indian petroleum pricing, PPAC, or fuel consumption
    using the document knowledge base. Use this for any question about these topics."""
    return rag_chain.invoke(question)

agent = create_agent(
    model=llm,
    tools=[search_petroleum_docs],
    system_prompt="You are a helpful assistant for petroleum sector questions. Use the search tool when relevant.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "session-1"}}

def ask(question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
    for m in result["messages"]:
        m.pretty_print()


In [31]:
ask("What does PPAC do?")

================================ Human Message =================================

What does PPAC do?
================================== Ai Message ==================================
Tool Calls:
  search_petroleum_docs (bd45891c-7432-4b7b-843c-8f8992402437)
 Call ID: bd45891c-7432-4b7b-843c-8f8992402437
  Args:
    question: What does PPAC do?
================================= Tool Message =================================
Name: search_petroleum_docs

PPAC compiles data on petroleum product consumption, pricing, and import/export trends across Indian states.
================================== Ai Message ==================================

PPAC stands for Petroleum Planning and Analysis Cell. It is a division of the Ministry of Petroleum and Natural Gas in India. PPAC compiles data on petroleum product consumption, pricing, and import/export trends across Indian states.
================================ Human Message =================================

Which ministry is it under?
=========

In [32]:
ask("Which ministry is it under?")

================================ Human Message =================================

What does PPAC do?
================================== Ai Message ==================================
Tool Calls:
  search_petroleum_docs (bd45891c-7432-4b7b-843c-8f8992402437)
 Call ID: bd45891c-7432-4b7b-843c-8f8992402437
  Args:
    question: What does PPAC do?
================================= Tool Message =================================
Name: search_petroleum_docs

PPAC compiles data on petroleum product consumption, pricing, and import/export trends across Indian states.
================================== Ai Message ==================================

PPAC stands for Petroleum Planning and Analysis Cell. It is a division of the Ministry of Petroleum and Natural Gas in India. PPAC compiles data on petroleum product consumption, pricing, and import/export trends across Indian states.
================================ Human Message =================================

Which ministry is it under?
=========

In [33]:
ask("What is the capital of France?")

================================ Human Message =================================

What does PPAC do?
================================== Ai Message ==================================
Tool Calls:
  search_petroleum_docs (bd45891c-7432-4b7b-843c-8f8992402437)
 Call ID: bd45891c-7432-4b7b-843c-8f8992402437
  Args:
    question: What does PPAC do?
================================= Tool Message =================================
Name: search_petroleum_docs

PPAC compiles data on petroleum product consumption, pricing, and import/export trends across Indian states.
================================== Ai Message ==================================

PPAC stands for Petroleum Planning and Analysis Cell. It is a division of the Ministry of Petroleum and Natural Gas in India. PPAC compiles data on petroleum product consumption, pricing, and import/export trends across Indian states.
================================ Human Message =================================

Which ministry is it under?
=========